<a href="https://colab.research.google.com/github/scottn66/255-final-project/blob/wp6-colab-gates/notebooks/07_input_blind_ab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Input-blind A/B on Colab

Runs the preprocess / PE-scale A/B from [`docs/INPUT-BLIND.md`](../docs/INPUT-BLIND.md) on a Colab GPU so the transformer can escape the background-only prior. **Runtime → Change runtime type → GPU, then Runtime → Run all.**

| Step | What | Stops when |
|---|---|---|
| Setup | clone this branch, install, mount Drive | no GPU / clone fails |
| Data | `scripts/get_data.sh` onto local disk (~24 GB) | md5 or split counts wrong |
| Diag | `scripts/diag_input_blind.py` (synthetic, no cubes) | ratios do not rise with `--standardize` |
| Smoke | `--model transformer --smoke --standardize` | smoke raises |
| A/B | 64 train / 8 val, ds2, B4, 4000 steps (~250 epochs), constant lr | (monitor only) |
| Summarize | first epoch with train FG mIoU ≥ 0.05, plus best val FG | |

Default grid (edit `AB_RUNS` to widen): three transformer arms at `lr 3e-4` (raw / standardize / standardize+PE 0.02) plus a CNN `--standardize` control. Full lr×preprocess grid is commented in Settings.

Checkpoints and `metrics.jsonl` go to Drive (`RUNS`). Cubes stay on `/content`. Re-Run all resumes unfinished tags via `sw.gates`.

AI-assisted notebook; Scott reviews every line.


## Settings


In [ ]:
import os
from pathlib import Path

BRANCH = os.environ.get("SW_BRANCH", "wp6-colab-gates")
REPO_URL = "https://github.com/scottn66/255-final-project.git"
RUNS = Path(os.environ.get("AB_RUNS_DIR", "/content/drive/MyDrive/sw_runs/input_blind_ab"))
WORK = Path(os.environ.get("AB_WORK", "/content/sw_ab"))
DATA_DIR = Path(os.environ.get("AB_DATA", "/content/data"))
AB_ZIP = os.environ.get("AB_ZIP", "")  # optional Drive copy of the zip
DELETE_ZIP = True
DEVICE = os.environ.get("AB_DEVICE", "cuda")
NUM_WORKERS = int(os.environ.get("AB_WORKERS", "2"))
POLL_SECONDS = int(os.environ.get("AB_POLL", "120"))

# 64 train / batch 4 => 16 steps/epoch; 250 epochs => 4000 optimizer steps.
AB_TRAIN_IMAGES = int(os.environ.get("AB_TRAIN_IMAGES", "64"))
AB_VAL_IMAGES = int(os.environ.get("AB_VAL_IMAGES", "8"))
AB_BATCH = int(os.environ.get("AB_BATCH", "4"))
AB_STEPS = int(os.environ.get("AB_STEPS", "4000"))
STEPS_PER_EPOCH = max(1, AB_TRAIN_IMAGES // AB_BATCH)
AB_EPOCHS = int(os.environ.get("AB_EPOCHS", str(AB_STEPS // STEPS_PER_EPOCH)))

# Shared train flags for every A/B arm (docs/INPUT-BLIND.md).
BASE = [
    "--downsample", "2",
    "--batch-size", str(AB_BATCH),
    "--max-train-images", str(AB_TRAIN_IMAGES),
    "--max-val-images", str(AB_VAL_IMAGES),
    "--no-cosine",  # constant lr
    "--num-workers", str(NUM_WORKERS),
]

# Priority grid at lr 3e-4. Uncomment the 1e-3 arms for the full INPUT-BLIND grid.
AB_RUNS = {
    "tf_raw_lr3e4": ["--model", "transformer", "--lr", "3e-4", *BASE],
    "tf_std_lr3e4": ["--model", "transformer", "--lr", "3e-4", "--standardize", *BASE],
    "tf_std_pe002_lr3e4": [
        "--model", "transformer", "--lr", "3e-4",
        "--standardize", "--pe-init-std", "0.02", *BASE,
    ],
    "cnn_std_lr1e3": ["--model", "cnn", "--lr", "1e-3", "--standardize", *BASE],
    # "tf_raw_lr1e3": ["--model", "transformer", "--lr", "1e-3", *BASE],
    # "tf_std_lr1e3": ["--model", "transformer", "--lr", "1e-3", "--standardize", *BASE],
    # "tf_std_pe002_lr1e3": [
    #     "--model", "transformer", "--lr", "1e-3",
    #     "--standardize", "--pe-init-std", "0.02", *BASE,
    # ],
}

WAIT_FOR_AB = True
SKIP_DIAG = False  # True: skip synthetic diag after a known-good session


## Setup


In [ ]:
import json
import shutil
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
T_START = time.time()


def sh(cmd):
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, check=True)


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO = Path("/content/255-final-project")
    if not REPO.is_dir():
        sh(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)])
    else:
        sh(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH])
        try:
            sh(["git", "-C", str(REPO), "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"])
        except subprocess.CalledProcessError as err:
            raise RuntimeError(
                f"{REPO} has local changes that block switching to {BRANCH}; "
                "delete the folder and Run all again"
            ) from err
    sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")])
else:
    _env = os.environ.get("REPO_ROOT", "").strip()
    _here = Path.cwd().resolve()
    REPO = Path(_env) if _env else next(
        p for p in [_here, *_here.parents] if (p / "src" / "sw" / "gates.py").is_file()
    )

os.chdir(REPO)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import torch
from sw import gates

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime → Change runtime type → GPU, then Run all again.")

COMMIT = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO, capture_output=True, text=True
).stdout.strip()
ON_BRANCH = subprocess.run(
    ["git", "branch", "--show-current"], cwd=REPO, capture_output=True, text=True
).stdout.strip()
ENV = {**os.environ, "PYTHONPATH": str(REPO / "src")}
RUNS.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else DEVICE

print(f"repo   {REPO} @ {COMMIT} (branch {ON_BRANCH or 'detached'})")
print(f"torch  {torch.__version__} on {GPU}")
print(f"A/B    {AB_EPOCHS} epochs × ~{STEPS_PER_EPOCH} steps = ~{AB_EPOCHS * STEPS_PER_EPOCH} optimizer steps")
print(f"runs   {RUNS}\nwork   {WORK}\ndata   {DATA_DIR}")


## Data


In [ ]:
from sw.data import SPLIT_COUNTS, SpectralWasteDataset


def counts_ok(root):
    try:
        return all(
            len(SpectralWasteDataset(root, split=s, random_flip=False)) == SPLIT_COUNTS[s]
            for s in ("train", "val")
        )
    except (FileNotFoundError, ValueError, RuntimeError):
        return False


SW_ROOT = Path(os.environ["SW_ROOT"]) if os.environ.get("SW_ROOT") else DATA_DIR / "spectralwaste_segmentation"
ALLOW_PARTIAL = os.environ.get("AB_ALLOW_PARTIAL") == "1"
if ALLOW_PARTIAL and (SW_ROOT / "hyper").is_dir():
    print(f"SW_ROOT {SW_ROOT}: partial data allowed (local test)")
elif counts_ok(SW_ROOT):
    print(f"SW_ROOT {SW_ROOT}: unpacked, train and val counts match")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(DATA_DIR).free / 2**30
    print(f"local disk free: {free:.0f} GiB (zip 21 GB + cubes 24 GB while unzipping)")
    if free < 50:
        print("WARNING: under 50 GiB free; the unzip may run out of space")
    zip_path = DATA_DIR / "spectralwaste_segmentation.zip"
    if AB_ZIP and Path(AB_ZIP).is_file() and not zip_path.is_file():
        sh(["cp", AB_ZIP, str(zip_path)])
    sh(["bash", str(REPO / "scripts" / "get_data.sh"), str(DATA_DIR)])
    SW_ROOT = DATA_DIR / "spectralwaste_segmentation"
    if not counts_ok(SW_ROOT):
        raise RuntimeError(f"{SW_ROOT}: the train/val counts do not match {SPLIT_COUNTS}")
    if DELETE_ZIP:
        zip_path.unlink(missing_ok=True)
os.environ["SW_ROOT"] = ENV["SW_ROOT"] = str(SW_ROOT)


## Diagnostic (synthetic)

No cubes required. Spatial-std/rms at tokenizer and head input should rise with `--standardize` and again with `--pe-init-std 0.02` (see `docs/INPUT-BLIND.md`).


In [ ]:
if SKIP_DIAG:
    print("diag skipped")
else:
    for flags in ([], ["--standardize"], ["--standardize", "--pe-init-std", "0.02"]):
        cmd = [sys.executable, str(REPO / "scripts" / "diag_input_blind.py"), *flags]
        print("$", " ".join(cmd), flush=True)
        out = subprocess.run(cmd, capture_output=True, text=True, env=ENV, cwd=REPO)
        print(out.stdout[-1500:] or out.stderr[-1500:])
        if out.returncode != 0:
            raise RuntimeError("diag_input_blind.py failed")


## Smoke with `--standardize`

Wiring check only. Must print `BandTransformer parameters: 488,487 (0.488M)`.


In [ ]:
smoke = subprocess.run(
    [
        sys.executable, "-m", "sw.train",
        "--model", "transformer", "--smoke", "--device", DEVICE,
        "--data-root", str(SW_ROOT), "--standardize",
        "--out-dir", str(WORK / "smoke_std"),
    ],
    capture_output=True, text=True, env=ENV, cwd=REPO,
)
print(smoke.stdout[-2000:], smoke.stderr[-2000:], sep="\n")
if smoke.returncode != 0:
    raise RuntimeError("smoke with --standardize failed")
if "BandTransformer parameters: 488,487 (0.488M)" not in smoke.stdout:
    raise RuntimeError("parameter line is not 488,487")
print("smoke passed")


## Launch A/B runs

Each tag is a background `python -m sw.gates run` with checkpoints on Drive. Interrupting a cell does not stop them. Re-running resumes from `last.pt`.


In [ ]:
PROCS = {}
for tag, flags in AB_RUNS.items():
    args = [
        *flags,
        "--epochs", str(AB_EPOCHS),
        "--device", DEVICE,
        "--data-root", str(SW_ROOT),
    ]
    PROCS[tag] = gates.launch(RUNS / tag, args, env=ENV)


## Monitor + escape summary


In [ ]:
def status_line(tag):
    out = RUNS / tag
    state = gates.run_state(out)
    proc = PROCS.get(tag)
    if proc is not None and proc.poll() is None:
        state = "running"
    try:
        rows = gates.read_metrics(out / "metrics.jsonl")
    except (FileNotFoundError, ValueError):
        return state, f"{tag:28s} {state:8s} no epoch finished yet"
    summary = gates.summarize_escape(rows)
    last = rows[-1]
    return state, (
        f"{tag:28s} {state:8s} epoch {last['epoch']:3d}/{AB_EPOCHS}  "
        f"loss {last['train_loss']:.4f}  train FG {last.get('train_fg_miou') or 0:.3f}  "
        f"val FG {last.get('val_fg_miou') or 0:.3f}  |  "
        + gates.format_escape_line("", summary).strip()
    )


while True:
    report = {tag: status_line(tag) for tag in AB_RUNS}
    print(time.strftime("%H:%M:%S"), *[line for _, line in report.values()], sep="\n  ", flush=True)
    states = [state for state, _ in report.values()]
    if all(state == "done" for state in states) or not WAIT_FOR_AB:
        break
    stopped = [
        tag for tag, (st, _) in report.items()
        if st not in ("done", "running") and gates.run_state(RUNS / tag) == "failed"
    ]
    if stopped:
        raise RuntimeError(f"A/B runs failed: {stopped}. Check {RUNS}/<tag>/train.log")
    time.sleep(POLL_SECONDS)

print("\n=== escape summary (train FG mIoU >= 0.05) ===")
for tag in AB_RUNS:
    out = RUNS / tag
    try:
        summary = gates.summarize_escape(gates.read_metrics(out / "metrics.jsonl"))
    except (FileNotFoundError, ValueError) as err:
        print(f"{tag}: no metrics yet ({err})")
        continue
    print(gates.format_escape_line(tag, summary))

print(f"\nelapsed {(time.time() - T_START) / 3600:.2f} h")
print("Next: pick the fastest escaping transformer arm; run a 30-epoch full-split screen:")
print(
    "  PYTHONPATH=src python -m sw.train --model transformer --downsample 2 "
    "--epochs 30 --batch-size 4 --num-workers 2 --device cuda --no-cosine "
    "--lr 3e-4 --standardize --pe-init-std 0.02 "
    f"--out-dir {RUNS}/tf_screen_std_pe002"
)


## What to send back

* Escape lines above (or `PYTHONPATH=src python -m sw.gates summarize $RUNS/*`).
* `RUNS/<tag>/metrics.jsonl` for each arm.
* Note whether standardisation was used (deviates from the paper's `[0, 1]` scaling).

Gates 0–2 in `06_transformer_gates.ipynb` are still the wiring path if this branch is new to the GPU. Optimiser warmup / grad-clip is still a separate PR.
